# Scratch Model With Qwen Tokenizer

This notebook trains this repo's scratch `TinyTransformer` using the local Qwen tokenizer from `models/qwen3-0.6b`.

This does not use Qwen weights. Only the tokenizer comes from Qwen.

## 1. Find The Repo Root And Use The Current Kernel

In [2]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

cwd = Path.cwd().resolve()
project_dir = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "step9_train_tiny_transformer.py").exists():
        project_dir = candidate
        break

if project_dir is None:
    raise RuntimeError("Could not find the repo root. Start VS Code from this project folder.")

os.chdir(project_dir)
print(f"project_dir: {project_dir}")
print(f"kernel python: {sys.executable}")

def run_py(script: str, *args: str) -> None:
    cmd = [sys.executable, "-u", script, *args]
    print(">", shlex.join(cmd))
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)


project_dir: C:\Users\kuchris\Desktop\github project\llm
kernel python: c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe


## 2. Check CUDA And Local Qwen Tokenizer

In [3]:
import torch
from pathlib import Path

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"cuda device: {torch.cuda.get_device_name(0)}")

qwen_dir = Path("models/qwen3-0.6b")
print(f"qwen tokenizer exists: {(qwen_dir / 'tokenizer.json').exists()}")
print(f"qwen config exists: {(qwen_dir / 'config.json').exists()}")


torch: 2.11.0+cu128
cuda available: True
cuda device: NVIDIA GeForce RTX 4080 Laptop GPU
qwen tokenizer exists: True
qwen config exists: True


## 3. Optional: Compare Tokenizers

In [4]:
run_py(
    "step26_compare_tokenizers.py",
    "--text",
    "Where is Hong Kong?",
    "--limit",
    "20",
)


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step26_compare_tokenizers.py --text 'Where is Hong Kong?' --limit 20
== Tokenizers ==
repo BPE: tokenizers/free_bpe_6144.json vocab=6,144
qwen: models/qwen3-0.6b vocab=151,669

== Sample 1 ==
text: 'She go to school yesterday.'
repo BPE tokens: 10
repo BPE ids: [52, 259, 931, 289, 1446, 488, 278, 340, 1680, 15]
repo BPE round trip: True
qwen tokens: 6
qwen ids: [7941, 728, 311, 2906, 13671, 13]
qwen round trip: True
repo/qwen token count ratio: 1.67x

== Sample 2 ==
text: 'Where is Hong Kong?'
repo BPE tokens: 8
repo BPE ids: [56, 2675, 363, 341, 418, 438, 418, 32]
repo BPE round trip: True
qwen tokens: 5
qwen ids: [9064, 374, 19180, 18211, 30]
qwen round trip: True
repo/qwen token count ratio: 1.60x

== Sample 3 ==
text: 'Correct the grammar, spelling, and punctuation mistakes in the following text.'
repo BPE tokens: 22
repo BPE ids: [36, 272, 4508, 262, 866, 312, 2252, 13, 470, 2095, 13, 288, 4153, 381, 2684,

## 4. Prepare Data If Needed

Skip this if you already have `data/wikitext103_clean.txt` and `data/dolly_train_eos_sft.txt`.

In [ ]:
run_py("download/step12_download_wikitext2.py")
run_py("step15_prepare_wikitext2.py")
run_py("step20_prepare_dolly.py")


## 5. Smoke Train With Qwen Tokenizer

This only checks that training starts. It creates a large checkpoint because the Qwen vocab is large.

In [ ]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "qwen_tokenizer_tiny_pretrain",
    "--device",
    "auto",
    "--max-iters",
    "2",
    "--save-every",
    "0",
    "--checkpoint",
    "checkpoints/qwen_tokenizer_smoke/tiny_transformer.pt",
)


## 6. Pretrain With Qwen Tokenizer

Use `--resume` if you stop and continue later.

In [5]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "qwen_tokenizer_tiny_pretrain",
    "--device",
    "auto",
)


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step9_train_tiny_transformer.py --preset qwen_tokenizer_tiny_pretrain --device auto
device: cuda
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (228407 > 131072). Running this sequence through the model will result in indexing errors
step     0 /  2000 | loss 12.1306
step    20 /  2000 | loss 8.5359
step    40 /  2000 | loss 8.0992
step    60 /  2000 | loss 7.3392
step    80 /  2000 | loss 7.7959
step   100 /  2000 | loss 8.1069
step   120 /  2000 | loss 7.5402
step   140 /  2000 | loss 7.9320
step   160 /  2000 | loss 6.9575
step   180 /  2000 | loss 7.1416
step   200 /  2000 | loss 6.9513
autosaved checkpoint: checkpoints\qwen_tokenizer_tiny_pretrain\tiny_transformer.pt
step   220 /  2000 | loss 6.3530
step   240 /  2000 | loss 6.7517
step   260 /  2000 | loss 6.7943
step   280 /  2000 | loss 6.4905
step   300 /  2000 | loss 7.3107
step   320 /

## 7. SFT With Qwen Tokenizer

In [6]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "qwen_tokenizer_tiny_sft",
    "--device",
    "auto",
)


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step9_train_tiny_transformer.py --preset qwen_tokenizer_tiny_sft --device auto
device: cuda
loaded init checkpoint: checkpoints\qwen_tokenizer_tiny_pretrain\tiny_transformer.pt
step     0 /  1000 | loss 8.1469
step    20 /  1000 | loss 8.0915
step    40 /  1000 | loss 8.7466
step    60 /  1000 | loss 5.3710
step    80 /  1000 | loss 6.9286
step   100 /  1000 | loss 7.6356
step   120 /  1000 | loss 7.0798
step   140 /  1000 | loss 0.0000
step   160 /  1000 | loss 7.1335
step   180 /  1000 | loss 6.9016
step   200 /  1000 | loss 8.3384
autosaved checkpoint: checkpoints\qwen_tokenizer_tiny_sft\tiny_transformer.pt
step   220 /  1000 | loss 6.0905
step   240 /  1000 | loss 6.6154
step   260 /  1000 | loss 3.5869
step   280 /  1000 | loss 3.9931
step   300 /  1000 | loss 5.3666
step   320 /  1000 | loss 6.3423
step   340 /  1000 | loss 6.7183
step   360 /  1000 | loss 7.8280
step   380 /  1000 | loss 5.0591
step   40

## 8. Generate From Scratch Model

In [10]:
run_py(
    "step10_generate.py",
    "--preset",
    "qwen_tokenizer_tiny_sft",
    "--device",
    "auto",
    "--prompt",
    "Where is Hong Kong?",
    "--max-new-tokens",
    "80",
    "--temperature",
    "0.9",
)


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step10_generate.py --preset qwen_tokenizer_tiny_sft --device auto --prompt 'Where is Hong Kong?' --max-new-tokens 80 --temperature 0.9
device: cuda
### Instruction:
Where is Hong Kong?

### Response:
Chile is a entirely variant that bills used by Original reflexball, it is considered 220 goals. The provider samples of mechanics is no more than one-mile-wide components used by short, and the kinds of 30 percent talks with a.net.<eos>

 type and mutation is a safe (Northern Ireland), and a planetDr.<eos>

 are built to avoid curtain designer) in


In [9]:
run_py(
    "step24_generate_qwen3.py",
    "--device",
    "auto",
    "--prompt",
    "Where is Hong Kong?",
)


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step24_generate_qwen3.py --device auto --prompt 'Where is Hong Kong?'
device: cuda

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 9513.67it/s]
Hong Kong is a special administrative region of China, located on the southern coast of China. It is a politically and economically important region with a unique status. The city of Hong Kong is known for its culture, economy, and international influence.
